## Matrix-vector multiplication

Consider the problem
\begin{equation}
y=Ax,
\end{equation}
where
\begin{equation}
A\in\mathbb R^{n\times n},
\qquad x,y\in\mathbb R^n.
\end{equation}

This is one of the most fundamental operations in numerical computing. It appears in:
- Iterative methods for solving linear systems.
- Numerical discretization of PDEs.
- Eigenvalue algorithms.
- Optimization algorithms.
- Time-dependent numerical simulations.

The mathematical operation is
\begin{equation}
y_i=\sum_{j=1}^{n}A_{ij}x_j,
\qquad i=1,\ldots,n.
\end{equation}

There are two nested summations, so the computational complexity is
\begin{equation}
O(n^2).
\end{equation}

### Implementation 1: Explicit Python loops

In [2]:
import numpy as np

def matvec_loop(A, x):
    n = len(x)
    y = np.zeros(n)

    for i in range(n):
        for j in range(n):
            y[i] += A[i, j] * x[j]

    return y

### Implementation 2: NumPy

In [3]:
def matvec_numpy(A, x):
    return A @ x
    

### Time both the implementations

In [4]:
import time

n = 1000

A = np.random.rand(n, n)
x = np.random.rand(n)

# Python implementation
start = time.perf_counter()
y1 = matvec_loop(A, x)
t1 = time.perf_counter() - start

# NumPy implementation
start = time.perf_counter()
y2 = matvec_numpy(A, x)
t2 = time.perf_counter() - start

print("Python loops:", t1)
print("NumPy:", t2)
print("Speedup:", t1/t2)

print("Difference:", np.linalg.norm(y1-y2))

Python loops: 0.41121984098572284
NumPy: 0.002104348037391901
Speedup: 195.4143676230396
Difference: 6.750380367908292e-12


- Actual timings depend on hardware, NumPy's underlying libraries and system load.

- Both implementations have the same asymptotic complexity, $\mathcal{O}(n^2)$.
- NumPy is generally much faster because its array operations execute in optimized compiled code.
- Python loops have substantial interpreter overhead.
- The mathematical algorithm has not changed, but the implementation has.

- `Python loops:` Each iteration involves Python interpreter overhead, including dynamic type checking and object handling.


- `NumPy operations:` When we write A @ x, Python calls NumPy's compiled implementation.
- The actual numerical computation is performed in optimized native code, largely avoiding Python's per-element overhead.

- In the first case, Python manages the individual iterations.
- In the second case, Python delegates the computation to compiled numerical routines.

`An important distinction`
- NumPy is not necessarily faster because it uses a better algorithm.
- Both implementations have computational complexity:
\begin{equation}
\mathcal{O}(n^2).
\end{equation}
- The difference is primarily in the execution overhead and optimization of the underlying implementation.

## Why does performance become important?

Consider solving a two-dimensional Poisson equation:
\begin{equation}
-\Delta u=f \quad \text{in } \Omega,
\end{equation}
using a finite difference discretization.

If we use $N$ grid points in each spatial direction, the number of unknowns is approximately
\begin{equation}
n=N^2.
\end{equation}
For a dense matrix-vector multiplication, the computational cost is
\begin{equation}
\mathcal{O}(n^2)=\mathcal{O}(N^4).
\end{equation}
- However, the discretized Laplacian has a sparse structure.
- Exploiting that structure can reduce the cost of a matrix-vector multiplication to approximately
  \begin{equation}
\mathcal{O}(n)=\mathcal{O}(N^2).
\end{equation}
This is a much more significant improvement than merely replacing Python loops with NumPy.

| Grid points per direction |  Unknowns |
| :--- | :---: |
| 10 | 100 |
| 100| 10,000 |
|500| 250,000|
|1000| 1,000,000|

### Should we rather optimize the implementation of a dense matrix-vector multiplication,
###                               or 
### exploit the sparse structure of the discretized operator?

- both matter, but algorithmic improvements can have a much greater impact.

`distinction`
- `Algorithmic optimization:` choosing a better computational method.
- `Implementation optimization:` implementing a given method more efficiently.


![image info](cs_wf.png)

`Summary`
- Performance optimization is not about making every line of Python code shorter or more clever.
- It is about understanding:
    - the computational work being performed,
    - identifying what costs the most, and
    - finding a mathematically sound way to reduce that cost.

## Benchmarking

### What is benchmarking?

Benchmarking is the systematic measurement and comparison of computational performance.

We may want to answer questions such as:
- Which implementation is faster?
- How does execution time depend on problem size?
- Does vectorization improve performance?
- Does a new implementation actually improve the original one?
- How much memory does an algorithm require?

`Two primary performance measures`:

`Execution time`: How long does a computation take?

`Memory consumption`: How much memory does the computation require?

### `Measuring execution time`

Python's `time` module.

There are two commonly encountered functions:

- `time.time()`
- `time.perf_counter()`

For benchmarking, prefer `time.perf_counter()` because it provides a high-resolution performance counter suitable for measuring short time intervals.

`three steps`:
1. Record the starting time.
2. Execute the computational task.
3. Record the finishing time.

The execution time is
\begin{equation}
T=t_{\mathrm{end}}-t_{\mathrm{start}}.
\end{equation}

The measured time may be affected by:

- Other processes running on the computer.
- CPU scheduling.
- Memory allocation.
- Cache effects.
- Initial library overhead.

Therefore, a single measurement is not necessarily representative.

### `Repeated measurements and statistical variation`

In [8]:
def benchmark(func, repeats=10):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        func()
        end = time.perf_counter()

        times.append(end - start)

    return np.array(times)

- Apply it to matrix-vector multiplication.

In [14]:
n = 1000

A = np.random.rand(n, n)
x = np.random.rand(n)

times = benchmark(lambda: matvec_numpy(A, x), repeats=20)

times = benchmark(lambda: A@x, repeats=20)

print("Mean:", np.mean(times))
print("Median:", np.median(times))
print("Standard deviation:", np.std(times))

Mean: 0.0029904072143835947
Median: 0.0029983859858475626
Standard deviation: 4.107569590278374e-05
